# Agent SDK solution

This notebook generates testdata using the agent sdk and your claude code subscription

You must be logged in to execute the script

In [ ]:
# Install dependencies

%pip install claude-agent-sdk

In [ ]:
import os
os.environ.pop("ANTHROPIC_API_KEY", None)

if os.environ.get("ANTHROPIC_API_KEY"):
    raise EnvironmentError(
        "ANTHROPIC_API_KEY is set in the environment. "
        "This notebook uses the Claude Agent SDK which authenticates via Claude Code — "
        "remove ANTHROPIC_API_KEY to avoid additional costs."
    )

In [ ]:
import json
import re
from claude_agent_sdk import query, ClaudeAgentOptions

# Execute claude code from a jupyter notebook

In [ ]:
import asyncio
import anyio
import sys
import threading

def run_async(coro):
    """Run async code in a background thread via anyio.run().

    Two problems in Jupyter on Windows:
    1. asyncio.run() fails because Jupyter already has a running event loop.
    2. The Jupyter event loop is a SelectorEventLoop which cannot spawn subprocesses.
       ProactorEventLoop is required; anyio's loop_factory backend option sets it.
    anyio.run() also sets up the sniffio context that the Agent SDK requires.
    """
    result_container = [None]
    error_container = [None]

    async def _wrapper():
        result_container[0] = await coro

    def thread_fn():
        backend_options = {}
        if sys.platform == "win32":
            backend_options["loop_factory"] = asyncio.ProactorEventLoop
        try:
            anyio.run(_wrapper, backend="asyncio", backend_options=backend_options)
        except Exception as e:
            error_container[0] = e

    t = threading.Thread(target=thread_fn, daemon=True)
    t.start()
    t.join()

    if error_container[0] is not None:
        raise error_container[0]
    return result_container[0]

# Generate Testdata

In [ ]:
def extract_json(text):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if match:
        return json.loads(match.group(1).strip())
    match = re.search(r'\[[\s\S]*\]', text)
    if match:
        return json.loads(match.group(0))
    raise ValueError(f'Could not extract JSON from response: {text[:200]}')

async def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing a task that requires Python, JSON, or a Regex to complete.

Output ONLY a valid JSON array with no surrounding text, no markdown fences, no explanation.

Format:
[
  {
    "task": "Description of task"
  }
]

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    options = ClaudeAgentOptions(
        model="claude-haiku-4-5" ,
        max_turns=1,
        allowed_tools=[]
    )
    text_output = ""
    async for message in query(prompt=prompt, options=options):
        if hasattr(message, "result"):
            text_output = message.result
    return extract_json(text_output)

In [ ]:
dataset = run_async(generate_dataset())

print(dataset)

# Write testdata

In [ ]:
import os
os.makedirs("010", exist_ok=True)
with open("010/dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)